In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Build a .py script that takes a snapshot date, trains a model and outputs artefact into storage.

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/17 10:43:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [4]:
model_train_date_str = os.getenv("MODEL_TRAIN_DATE", "2024-09-01")
train_test_period_months = int(os.getenv("TRAIN_TEST_MONTHS", 12))
oot_period_months       = int(os.getenv("OOT_MONTHS", 2))
train_test_ratio        = float(os.getenv("TRAIN_TEST_RATIO", 0.8))

config = {}
config["model_train_date_str"]    = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"]        = oot_period_months

config["model_train_date"] = datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"]     = config["model_train_date"] - timedelta(days=1)
config["oot_start_date"]   = config["model_train_date"] - relativedelta(months=oot_period_months)
config["train_test_end_date"]   = config["oot_start_date"] - timedelta(days=1)
config["train_test_start_date"] = config["oot_start_date"] - relativedelta(months=train_test_period_months)
config["train_test_ratio"]       = train_test_ratio

# helper to format back to ISO strings for Spark filters
def fmt(d): 
    return d.strftime("%Y-%m-%d")

# derive all split dates
train_start = config["train_test_start_date"]
train_end   = config["train_test_end_date"]
val_date    = train_end  + relativedelta(months=1)
test_date   = val_date   + relativedelta(months=1)
oot1_date   = config["oot_start_date"]
oot2_date   = config["oot_end_date"]
prod_nov    = oot2_date  + relativedelta(months=1)
prod_dec    = prod_nov   + relativedelta(months=1)

## get label store

In [5]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [6]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

# your “cutoff” for model training
mtd = datetime.strptime(config["model_train_date_str"], "%Y-%m-%d")

# TRAIN/VAL/TEST window is the 12 months immediately preceding the 2-month OOT window
# so your TRAIN window starts 14 months before mtd
train_start = mtd - relativedelta(months=config["oot_period_months"] + config["train_test_period_months"])
# TRAIN runs 12 months from there
train_end   = train_start + relativedelta(months=config["train_test_period_months"] - 1)

# the next three single‐month dates
val_date  = train_end + relativedelta(months=1)
test_date = val_date    + relativedelta(months=1)
oot1_date = test_date   + relativedelta(months=1)
oot2_date = oot1_date   + relativedelta(months=1)

# two production months
prod_nov_date = oot2_date + relativedelta(months=1)
prod_dec_date = prod_nov_date + relativedelta(months=1)

# helper to format for Spark filters
def fmt(d): return d.strftime("%Y-%m-%d")

from pyspark.sql.functions import col

labels = {
    "TRAIN": label_store_sdf.filter(
        (col("snapshot_date") >= fmt(train_start)) &
        (col("snapshot_date") <= fmt(train_end))
    ),
    "VAL":   label_store_sdf.filter(col("snapshot_date") == fmt(val_date)),
    "TEST":  label_store_sdf.filter(col("snapshot_date") == fmt(test_date)),
    "OOT1":  label_store_sdf.filter(col("snapshot_date") == fmt(oot1_date)),
    "OOT2":  label_store_sdf.filter(col("snapshot_date") == fmt(oot2_date)),
    "PROD_NOV": label_store_sdf.filter(col("snapshot_date") == fmt(prod_nov_date)),
    "PROD_DEC": label_store_sdf.filter(col("snapshot_date") == fmt(prod_dec_date)),
}

# sanity check
for name, sdf in labels.items():
    print(f"{name} labels:", sdf.count())

TRAIN labels: 5958
VAL labels: 485
TEST labels: 518
OOT1 labels: 511
OOT2 labels: 513
PROD_NOV labels: 491
PROD_DEC labels: 498


## get features

In [7]:
feature_location = "data/gold/feature/"

feature_path = "datamart/gold/feature/*.parquet"

features_store_sdf = (
    spark.read
         .option("mergeSchema", "true")   # optional, if you have evolving schemas
         .parquet(feature_path)
)

print("row_count:", features_store_sdf.count())
features_store_sdf.show(truncate=False)


row_count: 218902
+-----------+-------------+------+-----+-----+-----+-----+-----+------+------+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+------+------+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+-----------------+---------------+----------------------+--------------------+---------------------------+-----------------------------+-------------------------+---------------+-------------------------+-----------------------------+----------------------+-------------------+-------------------+-----------------+-------------------+------------------+-------------+----+----+----------+-------+-----------+
|Customer_ID|snapshot_date|fe_1  |fe_2 |fe_3 |fe_4 |fe_5 |fe_6 |fe_7  |fe_8  |fe_9 |fe_10|fe_1

In [8]:
feature_splits = {
    "TRAIN":   ((train_start, train_end)),
    "VAL":     ((val_date,    val_date)),
    "TEST":    ((test_date,   test_date)),
    "OOT1":    ((oot1_date,   oot1_date)),
    "OOT2":    ((oot2_date,   oot2_date)),
    "PROD_NOV":((prod_nov_date, prod_nov_date)),
    "PROD_DEC":((prod_dec_date, prod_dec_date)),
}

df_features = {}
for name, (start_dt, end_dt) in feature_splits.items():
    df_features[name] = features_store_sdf.filter(
        (col("snapshot_date") >= fmt(start_dt)) &
        (col("snapshot_date") <= fmt(end_dt))
    )
    print(f"{name} features:", df_features[name].count())

TRAIN features: 107688
VAL features: 9479
TEST features: 9517
OOT1 features: 9467
OOT2 features: 9430
PROD_NOV features: 9462
PROD_DEC features: 9489


## prepare data for modeling

In [9]:
from pyspark.sql import SparkSession, functions as F
from functools import reduce

spark = (
    SparkSession.builder
    .appName("JoinFeaturesLabels")
    .getOrCreate()
)

label_path = "datamart/gold/label_store/*.parquet"
labels_sdf = (
    spark.read
         .option("mergeSchema", "true")
         .parquet(label_path)
         .withColumn("snapshot_date", F.to_date("snapshot_date"))
)

feature_path = "datamart/gold/feature/*.parquet"
features_sdf = (
    spark.read
         .option("mergeSchema", "true")
         .parquet(feature_path)
         .withColumn("snapshot_date", F.to_date("snapshot_date"))
)

print("labels_sdf rows:  ", labels_sdf.count())
print("raw features_sdf rows:", features_sdf.count())

feature_cols = [c for c in features_sdf.columns if c.startswith("fe_")]

features_trimmed = features_sdf.select(
    "Customer_ID", "snapshot_date", *feature_cols
)

print("trimmed features_sdf rows:", features_trimmed.count(), "columns:", feature_cols)

# Left-join features onto labels
joined_sdf = labels_sdf.join(
    features_trimmed,
    on=["Customer_ID", "snapshot_date"],
    how="left"
)
print("after join:", joined_sdf.count())

nonnull_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNotNull() for c in feature_cols]
)

final_sdf = joined_sdf.filter(nonnull_condition)
print("after dropping all-null features:", final_sdf.count())

final_sdf.write.mode("overwrite").parquet("/data/final_merged/")
print("Wrote final merged table to /data/final_merged/")

labels_sdf rows:   8974
raw features_sdf rows: 218902
trimmed features_sdf rows: 218902 columns: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean']
after join: 8974
after dropping all-null features: 8974
Wrote final merged table to /data/final_merged/


In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("InspectMerged") \
    .getOrCreate()

final_sdf = spark.read.parquet("/data/final_merged/")
final_sdf.show(20, truncate=False)   # first 20 rows, no truncation

+-----------+-------------+---------------------+-----+----------+-----+-----+-----+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+------+-----+------+------+------+------+-----+-----------+
|Customer_ID|snapshot_date|loan_id              |label|label_def |fe_1 |fe_2 |fe_3 |fe_4 |fe_5 |fe_6 |fe_7  |fe_8 |fe_9 |fe_10|fe_11|fe_12|fe_13|fe_14 |fe_15|fe_16 |fe_17 |fe_18 |fe_19 |fe_20|fe_1_5_mean|
+-----------+-------------+---------------------+-----+----------+-----+-----+-----+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+------+-----+------+------+------+------+-----+-----------+
|CUS_0x10aa |2023-08-01   |CUS_0x10aa_2023_02_01|0    |30dpd_6mob|190.0|81.0 |152.0|111.0|189.0|65.0 |118.0 |202.0|-59.0|77.0 |200.0|89.0 |266.0|7.0   |261.0|227.0 |13.0  |-10.0 |-49.0 |65.0 |144.6      |
|CUS_0x113e |2023-08-01   |CUS_0x113e_2023_02_01|1    |30dpd_6mob|78.0 |8.0  |398.0|51.0 |49.0 |208.0|-62.0 |-30.0|155.0|246.0|82.0 |98.0 |37.0 |241.0 |183.0|158.0 |149.0 |52.0  |1

In [11]:
model_train_date_str  = os.getenv("MODEL_TRAIN_DATE", "2024-09-01")
train_test_period_mos = int(os.getenv("TRAIN_TEST_MONTHS", "12"))
oot_period_mos        = int(os.getenv("OOT_MONTHS",       "2"))
train_test_ratio      = float(os.getenv("TRAIN_TEST_RATIO", "0.8"))

model_train_date = datetime.strptime(model_train_date_str, "%Y-%m-%d")

oot_end_date        = model_train_date - timedelta(days=1)
oot_start_date      = model_train_date - relativedelta(months=oot_period_mos)
train_test_end_date = oot_start_date - timedelta(days=1)
train_test_start_date = train_test_end_date - relativedelta(months=train_test_period_mos) + timedelta(days=1)

def fmt(d): return d.strftime("%Y-%m-%d")

print("CONFIG DATES:")
print("  TRAIN/TEST window:", fmt(train_test_start_date), "→", fmt(train_test_end_date))
print("  OOT window:",        fmt(oot_start_date),       "→", fmt(oot_end_date))

spark = SparkSession.builder.appName("ModelSplits").getOrCreate()

final_sdf = (
    spark.read
         .parquet("/data/final_merged/")
         .withColumn("snapshot_date", F.to_date("snapshot_date"))
)
print("total rows in merged table:", final_sdf.count())

oot_sdf = final_sdf.filter(
    (col("snapshot_date") >= fmt(oot_start_date)) &
    (col("snapshot_date") <= fmt(oot_end_date))
)
train_test_sdf = final_sdf.filter(
    (col("snapshot_date") >= fmt(train_test_start_date)) &
    (col("snapshot_date") <= fmt(train_test_end_date))
)

print("rows in TRAIN/TEST window:", train_test_sdf.count())
print("rows in OOT window:", oot_sdf.count())

feature_cols = [c for c in final_sdf.columns if c.startswith("fe_")]
print("Using feature columns:", feature_cols)

train_sdf, test_sdf = train_test_sdf.randomSplit(
    [train_test_ratio, 1 - train_test_ratio],
    seed=88
)
print("X_train rows:", train_sdf.count())
print("X_test  rows:", test_sdf.count())
print("X_oot   rows:", oot_sdf.count())

for name, sdf in [("y_train", train_sdf), ("y_test", test_sdf), ("y_oot", oot_sdf)]:
    mean_lbl = sdf.agg(F.mean("label").alias("mean_label")).first()["mean_label"]
    print(f"{name}: count={sdf.count()}, mean_label={mean_lbl:.2f}")

CONFIG DATES:
  TRAIN/TEST window: 2023-07-01 → 2024-06-30
  OOT window: 2024-07-01 → 2024-08-31
total rows in merged table: 8974
rows in TRAIN/TEST window: 5958
rows in OOT window: 1003
Using feature columns: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean']
X_train rows: 4780
X_test  rows: 1178
X_oot   rows: 1003
y_train: count=4780, mean_label=0.28
y_test: count=1178, mean_label=0.28
y_oot: count=1003, mean_label=0.29


In [12]:
splits = {
  "TRAIN":   ((train_start, train_end)),
  "VAL":     ((val_date,     val_date)),
  "TEST":    ((test_date,    test_date)),
  "OOT1":    ((oot1_date,    oot1_date)),
  "OOT2":    ((oot2_date,    oot2_date)),
}

df = final_sdf

model_train_date = datetime.strptime("2024-09-01", "%Y-%m-%d")
train_start = model_train_date - relativedelta(months=14)
train_end   = train_start + relativedelta(months=11)
val_date    = train_end + relativedelta(months=1)
test_date   = val_date   + relativedelta(months=1)
oot1_date   = test_date  + relativedelta(months=1)
oot2_date   = oot1_date  + relativedelta(months=1)

def fmt(d): return d.strftime("%Y-%m-%d")

splits = {
    "TRAIN":   (train_start, train_end),
    "VAL":     (val_date,    val_date),
    "TEST":    (test_date,   test_date),
    "OOT1":    (oot1_date,   oot1_date),
    "OOT2":    (oot2_date,   oot2_date),
}

df_splits = {}
for name, (start_dt, end_dt) in splits.items():
    df_splits[name] = df.filter(
        (col("snapshot_date") >= fmt(start_dt)) &
        (col("snapshot_date") <= fmt(end_dt))
    )

prod_nov = oot2_date + relativedelta(months=1)
prod_dec = prod_nov  + relativedelta(months=1)

df_splits["PROD_NOV"] = df.filter(col("snapshot_date") == fmt(prod_nov))
df_splits["PROD_DEC"] = df.filter(col("snapshot_date") == fmt(prod_dec))

for name, sdf in df_splits.items():
    print(name, "rows:", sdf.count())

TRAIN rows: 5958
VAL rows: 485
TEST rows: 518
OOT1 rows: 511
OOT2 rows: 513
PROD_NOV rows: 491
PROD_DEC rows: 498


## train model

In [13]:
# #!/usr/bin/env python3
# from pyspark.sql import SparkSession, functions as F
# from pyspark.ml import Pipeline
# from pyspark.ml.feature import (
#     VectorAssembler, StandardScaler, QuantileDiscretizer, ChiSqSelector
# )
# from pyspark.ml.classification import GBTClassifier
# from pyspark.ml.evaluation      import BinaryClassificationEvaluator
# from pyspark.ml.tuning         import CrossValidator, ParamGridBuilder

# spark = SparkSession.builder.appName("SparkMLPipeline").getOrCreate()

# df = (
#     spark.read
#          .parquet("/data/final_merged/")
#          .withColumn("label", F.col("label").cast("double"))
#          .withColumn("snapshot_date", F.to_date("snapshot_date"))
# )

# # define your train/val/test/oot splits exactly as before...
# # here we just take a single TRAIN slice for CV demonstration
# train_df = df.filter(F.col("snapshot_date") < "2024-09-01")


# feature_cols = [c for c in df.columns if c.startswith("fe_")]

# # Spark’s ChiSqSelector requires categorical features; we’ll bin each fe_* into 10 quantiles
# discretizers = [
#     QuantileDiscretizer(
#         numBuckets=10,
#         inputCol=c,
#         outputCol=c + "_binned",
#         handleInvalid="keep"
#     )
#     for c in feature_cols
# ]

# binned_cols = [c + "_binned" for c in feature_cols]


# assembler_raw = VectorAssembler(
#     inputCols=feature_cols,
#     outputCol="features_raw"
# )

# scaler = StandardScaler(
#     inputCol="features_raw",
#     outputCol="features_scaled",
#     withMean=True,
#     withStd=True
# )


# selector = ChiSqSelector(
#     numTopFeatures=19,            # your best_k
#     featuresCol="features_scaled",
#     labelCol="label",
#     outputCol="features_selected"
# )


# gbt = GBTClassifier(
#     labelCol="label",
#     featuresCol="features_selected",
#     maxIter=415,                  # n_estimators
#     maxDepth=3,
#     stepSize=0.009057293740640768,     # learning_rate
#     subsamplingRate=0.7276734855131275, # subsample
#     featureSubsetStrategy="0.6232189363962957", # colsample_bytree
#     seed=42
# )

# pipeline = Pipeline(stages=[
#     *discretizers,
#     assembler_raw,
#     scaler,
#     selector,
#     gbt
# ])

# evaluator = BinaryClassificationEvaluator(
#     labelCol="label",
#     metricName="areaUnderROC"
# )

# paramGrid = (
#     ParamGridBuilder()
#     # you can grid‐search over additional hyperparams if you like:
#     # .addGrid(gbt.maxDepth, [3, 5, 7])
#     # .addGrid(gbt.stepSize, [0.005, 0.01])
#     .build()
# )

# cv = CrossValidator(
#     estimator=pipeline,
#     estimatorParamMaps=paramGrid,
#     evaluator=evaluator,
#     numFolds=5,
#     parallelism=4
# )

# cvModel = cv.fit(train_df)

# bestAUC = evaluator.evaluate(cvModel.transform(train_df))
# print(f"▶ Best train ROC AUC (5-fold CV): {bestAUC:.4f}")

# import numpy as np
# featImp = cvModel.bestModel.stages[-1].featureImportances.toArray()
# # but we want the ChiSqSelector mask
# selectorModel = cvModel.bestModel.stages[-2]
# selectedIndices = selectorModel.selectedFeatures
# kept = [feature_cols[i] for i in selectedIndices]
# print(f"\nKept {len(kept)} features by ChiSqSelector:", kept)


In [14]:
!pip install pyarrow fastparquet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [25]:
## model best parameters were obtained previously and now refit again for training

#!/usr/bin/env python3
import os
import random
from datetime import datetime
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd

from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline          import Pipeline
from sklearn.metrics           import (
    roc_auc_score,
    fbeta_score,
    classification_report
)
from sklearn.model_selection   import train_test_split
import xgboost as xgb

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

data_path = "/data/final_merged/"
print(f"Loading merged data from {data_path} …")
df = pd.read_parquet(data_path, engine="pyarrow")
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

# ensure snapshot_date is a pandas datetime64[ns] column
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

train_start  = pd.Timestamp(2023,  7, 1)
train_end    = train_start + pd.DateOffset(months=11)   # → 2024-06-01
val_date     = train_end   + pd.DateOffset(months=1)    # → 2024-07-01
test_date    = val_date     + pd.DateOffset(months=1)    # → 2024-08-01
oot1_date    = test_date    + pd.DateOffset(months=1)    # → 2024-09-01
oot2_date    = oot1_date    + pd.DateOffset(months=1)    # → 2024-10-01
sim_nov_date = oot2_date    + pd.DateOffset(months=1)    # → 2024-11-01
sim_dec_date = sim_nov_date + pd.DateOffset(months=1)    # → 2024-12-01

print("Date splits:")
for name, dt in [
    ("TRAIN_START", train_start), ("TRAIN_END", train_end),
    ("VAL_DATE",   val_date),     ("TEST_DATE", test_date),
    ("OOT1",       oot1_date),    ("OOT2", oot2_date),
    ("PROD_NOV",   sim_nov_date), ("PROD_DEC", sim_dec_date),
]:
    print(f"  {name:10} → {dt.date()}")


train   = df[(df.snapshot_date >= train_start) & (df.snapshot_date <= train_end)]
val     = df[ df.snapshot_date == val_date ]
test    = df[ df.snapshot_date == test_date ]
oot1    = df[ df.snapshot_date == oot1_date ]
oot2    = df[ df.snapshot_date == oot2_date ]
prod_nov= df[ df.snapshot_date == sim_nov_date ]
prod_dec= df[ df.snapshot_date == sim_dec_date ]

print("Split counts:",
      len(train), "TRAIN /",
      len(val),   "VAL /",
      len(test),  "TEST /",
      len(oot1),  "OOT1 /",
      len(oot2),  "OOT2 /",
      len(prod_nov), "PROD_NOV /",
      len(prod_dec), "PROD_DEC"
)


drop_cols = ['Customer_ID','snapshot_date','label_def','loan_id','Name','SSN']
feature_cols = [c for c in train.columns if c not in drop_cols + ['label']]

print("Using features:", feature_cols)

X_train_full = train[feature_cols]
y_train_full = train['label']

X_val = val[feature_cols];   y_val = val['label']
X_test= test[feature_cols];  y_test= test['label']
X_oot = pd.concat([oot1, oot2])[feature_cols]
y_oot = pd.concat([oot1, oot2])['label']

test_ratio = 0.2
X_train, X_test2, y_train, y_test2 = train_test_split(
    X_train_full, y_train_full,
    test_size=test_ratio,
    random_state=SEED,
    shuffle=True,
    stratify=y_train_full
)

print("After stratified split – TRAIN:", X_train.shape[0], 
      "TEST:", X_test2.shape[0])

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="drop")

# scale_pos_weight to rebalance the classes
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

best_params = {
    "n_estimators":     415,
    "max_depth":        3,
    "learning_rate":    0.009057293740640768,
    "subsample":        0.7276734855131275,
    "colsample_bytree": 0.6232189363962957,
    "reg_alpha":        0.9502195804742386,
    "reg_lambda":       1.539615751287922,
    "eval_metric":      "logloss",
    "random_state":     SEED,
    "scale_pos_weight": scale_pos,
}

best_k = 19
beta   = 2.0
best_thresh = 0.32

pipe = Pipeline([
    ("pre",    preproc),
    ("select", SelectKBest(mutual_info_classif, k=best_k)),
    ("clf",    xgb.XGBClassifier(**best_params))
])

print("Fitting pipeline…")
pipe.fit(X_train, y_train)
print(f"Using fixed threshold for F{beta:.0f}: {best_thresh:.2f}")

def evaluate(name, X, y):
    proba = pipe.predict_proba(X)[:, 1]
    preds = (proba >= best_thresh).astype(int)
    auc   = roc_auc_score(y, proba)
    gini  = 2 * auc - 1
    f2    = fbeta_score(y, preds, beta=beta)
    print(f"\n{name:6} @ thresh={best_thresh:.2f}")
    print(f"  F{beta:.0f}: {f2:.4f}  AUC: {auc:.4f}  Gini: {gini:.4f}")
    print(classification_report(y, preds, digits=4))

for split_name, (X, y) in [
    ("TRAIN", (X_train, y_train)),
    ("VAL",   (X_val,   y_val)),
    ("TEST",  (X_test2, y_test2)),
    ("OOT",   (X_oot,   y_oot))
]:
    evaluate(split_name, X, y)

feat_names = pipe.named_steps["pre"].get_feature_names_out()
mask       = pipe.named_steps["select"].get_support()
kept       = feat_names[mask]
dropped    = feat_names[~mask]
print(f"\nKept {len(kept)} features:", kept.tolist())
print(f"Dropped {len(dropped)} features:", dropped.tolist())

Loading merged data from /data/final_merged/ …
Total rows: 8974
Columns: ['Customer_ID', 'snapshot_date', 'loan_id', 'label', 'label_def', 'fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean']
Date splits:
  TRAIN_START → 2023-07-01
  TRAIN_END  → 2024-06-01
  VAL_DATE   → 2024-07-01
  TEST_DATE  → 2024-08-01
  OOT1       → 2024-09-01
  OOT2       → 2024-10-01
  PROD_NOV   → 2024-11-01
  PROD_DEC   → 2024-12-01
Split counts: 5958 TRAIN / 485 VAL / 518 TEST / 511 OOT1 / 513 OOT2 / 491 PROD_NOV / 498 PROD_DEC
Using features: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean']
After stratified split – TRAIN: 4766 TEST: 1192
Fitting pipeline…
Using fixed threshold for F2: 0.32

TRAIN  @ thresh=0.32
  F2: 0.6823  AUC: 0.7413  Gi

In [13]:
model_artefact = {}

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

{'data_dates': {'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
                'model_train_date_str': '2024-09-01',
                'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
                'train_test_period_months': 12,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)},
 'data_stats': {'X_oot': 1003,
                'X_test': 1192,
                'X_train': 4766,
                'y_oot': np.float64(0.29),
                'y_test': np.float64(0.28),
                'y_train': np.float64(0.28)},
 'hp_params': {'colsample_bytree': 0.8,
               'gamma': 0.1,
               'learning_rate': 0.1,
               'max_depth': 3,
               'min_child_weight': 1,
               'n_estimators': 50,
        

## save artefact to model bank

In [14]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [ ]:
# Full path to the file
file_path = os.path.join(model_bank_directory,'my_xgb_pipeline.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)